In [1]:
import pandas as pd
from pathlib import Path

carriers_path = 'plants_data/carriers/'

belgium = pd.read_csv(Path(carriers_path) / 'Belgium_2024.csv', sep=',')
germany = pd.read_csv(Path(carriers_path) / 'Germany_2024.csv', sep=',')
netherlands = pd.read_csv(Path(carriers_path) / 'Netherlands_2024.csv', sep=',')
# keep colums MTU (CET/CEST), Day-ahead Price (EUR/MWh)
belgium = belgium[['MTU (CET/CEST)', 'Day-ahead Price (EUR/MWh)']]
germany = germany[['MTU (CET/CEST)', 'Day-ahead Price (EUR/MWh)']]
netherlands = netherlands[['MTU (CET/CEST)', 'Day-ahead Price (EUR/MWh)']]

# remove anything after the second space included
belgium['MTU (CET/CEST)'] = belgium['MTU (CET/CEST)'].str.split(' ').str[0:2].str.join(' ')
germany['MTU (CET/CEST)'] = germany['MTU (CET/CEST)'].str.split(' ').str[0:2].str.join(' ')
netherlands['MTU (CET/CEST)'] = netherlands['MTU (CET/CEST)'].str.split(' ').str[0:2].str.join(' ')
# convert MTU (CET/CEST) column to datetime format
belgium['MTU (CET/CEST)'] = pd.to_datetime(belgium['MTU (CET/CEST)'], format='%d/%m/%Y %H:%M:%S')
germany['MTU (CET/CEST)'] = pd.to_datetime(germany['MTU (CET/CEST)'], format='%d/%m/%Y %H:%M:%S')
netherlands['MTU (CET/CEST)'] = pd.to_datetime(netherlands['MTU (CET/CEST)'], format='%d/%m/%Y %H:%M:%S')

# if datetime are repeated, keep the first occurrence
belgium = belgium.drop_duplicates(subset=['MTU (CET/CEST)'], keep='first')
germany = germany.drop_duplicates(subset=['MTU (CET/CEST)'], keep='first')
netherlands = netherlands.drop_duplicates(subset=['MTU (CET/CEST)'], keep='first')

# if not hourly data, resample to hourly data by taking the mean of each hour
belgium = belgium.set_index('MTU (CET/CEST)').resample('h').mean().reset_index()
germany = germany.set_index('MTU (CET/CEST)').resample('h').mean().reset_index()
netherlands = netherlands.set_index('MTU (CET/CEST)').resample('h').mean().reset_index()

full_index = pd.date_range(start='2024-01-01 00:00:00', end='2024-12-31 23:00:00', freq='h')
# reindex to have a full year from 2024-01-01 to 2024-12-31 23:00:00
belgium = belgium.set_index('MTU (CET/CEST)').reindex(full_index).rename_axis('MTU (CET/CEST)').reset_index()
germany = germany.set_index('MTU (CET/CEST)').reindex(full_index).rename_axis('MTU (CET/CEST)').reset_index()
netherlands = netherlands.set_index('MTU (CET/CEST)').reindex(full_index).rename_axis('MTU (CET/CEST)').reset_index()

# output cleaned files
output_file = 'electricity_prices_2024.csv'
with open(Path(carriers_path) / output_file, 'w') as f:
    f.write('datetime,Belgium (EUR/MWh),Germany (EUR/MWh),Netherlands (EUR/MWh)\n')
    for i in range(len(full_index)):
        f.write(f"{full_index[i].strftime('%Y-%m-%d %H:%M:%S')},{belgium['Day-ahead Price (EUR/MWh)'].iloc[i]},{germany['Day-ahead Price (EUR/MWh)'].iloc[i]},{netherlands['Day-ahead Price (EUR/MWh)'].iloc[i]}\n")


Electricity data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'
scenario = 'flexible'
year = 2030

country_dict = {
    'Belgium': 'BE',
    'Germany': 'DE',
    'Netherlands': 'NL'}

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)
prices_data = pd.read_csv(f'plants_data/electricity_data/prices_AC_allin_{scenario}_{year}.csv', sep=',', index_col=0)
# Print prezzi per verifica
print(prices_data)

# --- Indice orario per il 2024 ---
full_index = pd.date_range(f'2024-01-01 00:00:00', f'2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
# with pd.ExcelWriter(Path(carriers_path) / 'electricity.xlsx', engine='openpyxl', mode='w') as writer:
#     for plant in europe_plants.index:
#         country = europe_plants.loc[plant, 'country']
#         country_code = country_dict[country]

#         # --- Prezzi ---
#         prices = prices_data[country_code].values
#         prices = np.clip(prices, 0.01, None)  # evitiamo negativi
#         print(prices)

#         # --- Creazione DataFrame per l’impianto ---
#         df = pd.DataFrame({
#             'import_price': prices,
#             'demand': 0.0,                # MW costante
#             'export_price': 0.01,         # EUR/MWh
#             'import_limit': 1e6,          # MW
#             'export_limit': 1e6           # MW
#         }, index=full_index)

#         # --- Salvataggio in foglio dedicato ---
#         df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')


IndexError: At least one sheet must be visible

Hydrogen data

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'hydrogen.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')


Ammonia

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'ammonia.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        demand = europe_plants.loc[plant, 'ammonia'] if 'ammonia' in europe_plants.columns else 0.0

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': demand/8760, # convertiamo in hourly demand
            'export_price': 0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')


Ethylene

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'ethylene.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        demand = europe_plants.loc[plant, 'ethylene'] if 'ethylene' in europe_plants.columns else 0.0

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': demand/8760, # convertiamo in hourly demand
            'export_price': 0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Methanol

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'methanol.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        demand = europe_plants.loc[plant, 'methanol'] if 'methanol' in europe_plants.columns else 0.0

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': demand/8760, # convertiamo in hourly demand
            'export_price': 0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Propylene

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'propylene.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        demand = europe_plants.loc[plant, 'propylene'] if 'propylene' in europe_plants.columns else 0.0

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': demand/8760, # convertiamo in hourly demand
            'export_price': 0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Carbon dioxide

In [34]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'CO2.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': 618, # considering DAC cost
            'demand': np.nan,
            'export_price': 0,
            'import_limit': 1e6,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')


methane

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'methane.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': 56,
            'demand': 0,
            'export_price': 0,
            'import_limit': 1e6,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Heat

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'heat.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Steam

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'steam.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

HBfeed

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'HBfeed.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0.0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

CO2captured

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'CO2captured.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:
        if plant == 'NL1':
            # --- Creazione DataFrame per l’impianto ---
            df = pd.DataFrame({
                'import_price': 100,
                'demand': 0.0,
                'export_price': -0.01,
                'import_limit': 1e6,
                'export_limit': 1e6
            }, index=full_index)
        else:
        # --- Creazione DataFrame per l’impianto ---
            df = pd.DataFrame({
                'import_price': np.nan,
                'demand': 0.0,
                'export_price': 0.0,
                'import_limit': 0,
                'export_limit': 1e6
            }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Nitrogen

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'nitrogen.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Naphtha

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'naphtha.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': 732,
            'demand': 0,
            'export_price': 0,
            'import_limit': 1e6,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Olefins

In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'olefins.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Crackergas

In [20]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'crackergas.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Syngas

In [28]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'syngas.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

MPW (Mixed Plastic Waste)

In [35]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'MPW.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': 780,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 1e6,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Ethanol

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'ethanol.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': 1734,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 1e6,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Propane

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2024 ---
full_index = pd.date_range('2024-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'propane.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': 1473,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 1e6,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')